# 04 导出 ONNX

从 checkpoint 导出 ONNX 模型，供前端 `OnnxPoseCorrector` 使用。模型输出 `corrected = landmarks + delta`，符合前端 I/O 约定。

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path("/content/ai_form_1")
DATA_ROOT = Path("/content/drive/MyDrive/ai_form_coach_data")
CKPT_PATH = DATA_ROOT / "checkpoints" / "pose_corrector_best.pt"
OUT_ONNX = DATA_ROOT / "pose_corrector.onnx"

sys.path.insert(0, str(PROJECT_ROOT / "tools" / "train_denoiser" / "src"))
from model import PoseCorrectorNet

In [ ]:
model = PoseCorrectorNet()
ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["model_state"])
model.eval()

In [ ]:
# 使用 forward_corrected 导出，输出 corrected（前端期望）
class ExportWrapper(torch.nn.Module):
  def __init__(self, net):
    super().__init__()
    self.net = net
  def forward(self, landmarks, phase):
    return self.net.forward_corrected(landmarks, phase)

wrapper = ExportWrapper(model)
dummy_landmarks = torch.randn(1, 33, 3)
dummy_phase = torch.rand(1, 1)

torch.onnx.export(
  wrapper,
  (dummy_landmarks, dummy_phase),
  str(OUT_ONNX),
  input_names=["landmarks", "phase"],
  output_names=["corrected", "confidence"],
  dynamic_axes={
    "landmarks": {0: "batch"},
    "phase": {0: "batch"},
    "corrected": {0: "batch"},
    "confidence": {0: "batch"},
  },
  opset_version=15,
)
print(f"已导出: {OUT_ONNX}")

In [ ]:
# 合并为单文件（无外部 .onnx.data），浏览器 onnxruntime-web 无法加载外部数据
import onnx
onnx_model = onnx.load(str(OUT_ONNX), load_external_data=True)
onnx.save_model(onnx_model, str(OUT_ONNX), save_as_external_data=False)
print("已保存为单文件（无外部数据），可直接用于浏览器")